# 04 - Selección de Variables

In [1]:
# Librerías
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Ruta de conexión

PROJECT_ROOT = Path().resolve().parents[1]

PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
PATH_MODEL = PROJECT_ROOT / "data" / "model"

df_model = pd.read_csv(PATH_PROCESSED / "features_intra_anuales_2007-2024.csv")
clusters_mun = pd.read_csv(PATH_MODEL / "clusters_municipales.csv")

In [3]:
# Merge clusters
df_model = df_model.merge(clusters_mun, on="municipio", how="left")
print(df_model["cluster"].value_counts())

cluster
2    198
1    162
3     54
0     36
Name: count, dtype: int64


In [4]:
# Selección de variables
target = "Rendimiento (t/ha)"

corr = df_model.corr(numeric_only=True)[target].abs().sort_values(ascending=False)
vars_seleccion = corr[corr > 0.14].index.tolist()
vars_seleccion.remove(target)

X = df_model[vars_seleccion]
y = df_model[target]

In [5]:
# Train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
# Lasso
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_train_scaled, y_train)

coef = pd.Series(lasso.coef_, index=X.columns)
vars_lasso = coef[coef != 0].sort_values(key=abs, ascending=False)

print(vars_lasso)

produccion_t                       1.283264e-01
temp_mean_z_mean_anual             9.222881e-02
ndvi_mean_min                     -5.985608e-02
ndvi_mean_sum_m04_06              -5.108147e-02
et_potencial_mm_std                4.757853e-02
ndvi_mean_std                     -4.666644e-02
ndvi_min_m12                       4.548664e-02
ndvi_mean_min_m01_06               3.971794e-02
flag_temp_media_alta_p90          -3.427216e-02
et_potencial_mm_m01               -2.803728e-02
precip_max_mm_m08                  2.617241e-02
racha_max_flag_deficit_alto_p90   -2.304492e-02
deficit_hidrico_mm_max_m01_03     -2.283915e-02
ndvi_mean_m05                      2.181217e-02
temp_max_max_m04_06                1.840277e-02
balance_hidrico_mm_min_m01_06     -1.833530e-02
ndvi_mean_min_m01_03              -1.761357e-02
ndvi_min_m01                       1.417764e-02
precip_max_mm_m07                  1.361388e-02
et_real_mm_std                     8.075959e-03
ndvi_mean_m02                      6.282

In [7]:
# Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

importancias = pd.Series(rf.feature_importances_, index=X.columns)
vars_rf_top = importancias.sort_values(ascending=False).head(20).index

print(importancias.sort_values(ascending=False).head(20))

produccion_t                     0.275995
ndvi_min_m12                     0.075173
et_potencial_mm_std              0.068227
temp_mean_z_mean_anual           0.051664
temp_mean_z_min_anual            0.044598
ndvi_mean_m05                    0.032951
et_potencial_mm_m01              0.027360
et_real_mm_z_min_anual           0.026773
temp_max_z_max_anual             0.025512
temp_mean_z_max_anual            0.025246
precip_max_mm_m07                0.024992
ndvi_min_m01                     0.024622
precip_max_mm_m08                0.024321
ndvi_mean_m01                    0.020935
ndvi_mean_m08                    0.019095
et_real_mm_std                   0.016976
balance_hidrico_mm_min_m01_06    0.015424
ndvi_mean_std                    0.014398
ndvi_mean_m02                    0.013467
temp_max_z_mean_anual            0.012957
dtype: float64


In [8]:
# Intersección final
vars_final = list(set(vars_lasso.index) & set(vars_rf_top))

print(f"Total variables seleccionadas: {len(vars_final)}")
print(vars_final)

Total variables seleccionadas: 13
['et_real_mm_std', 'precip_max_mm_m07', 'ndvi_mean_m05', 'balance_hidrico_mm_min_m01_06', 'produccion_t', 'ndvi_min_m01', 'et_potencial_mm_m01', 'ndvi_mean_std', 'et_potencial_mm_std', 'precip_max_mm_m08', 'ndvi_mean_m02', 'ndvi_min_m12', 'temp_mean_z_mean_anual']


In [9]:
# Guardar variables seleccionadas

ruta_vars = PATH_MODEL / "variables_seleccionadas.csv"
ruta_rf = PATH_MODEL / "importancia_variables_rf.csv"
ruta_lasso = PATH_MODEL / "coeficientes_lasso.csv"

pd.DataFrame({"variable": vars_final}).to_csv(ruta_vars, index=False)
importancias.sort_values(ascending=False).to_csv(ruta_rf)
vars_lasso.to_csv(ruta_lasso)

print("Variables guardadas:", ruta_vars)
print("Importancias RF guardadas:", ruta_rf)
print("Coeficientes Lasso guardados:", ruta_lasso)

Variables guardadas: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/model/variables_seleccionadas.csv
Importancias RF guardadas: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/model/importancia_variables_rf.csv
Coeficientes Lasso guardados: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/model/coeficientes_lasso.csv
